# Voice blurring demo (Cohen–Hadria et al., 2019, §2.2)

Load one audio file and run **low-pass blurring**, **MFCC inversion**, or **both in sequence**, using `voice_blurring.low_pass_blur` and `voice_blurring.mfcc_inversion_blur`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import librosa
import numpy as np
from IPython.display import Audio, display

# Repo root on sys.path (works if kernel cwd is repo root or notebooks/)
for _root in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (_root / "voice_blurring").is_dir():
        sys.path.insert(0, str(_root))
        break

from voice_blurring import low_pass_blur, mfcc_inversion_blur

# Set AUDIO_PATH to a WAV (or other librosa-readable) file, or leave unset for a short synthetic tone
_audio = os.environ.get("AUDIO_PATH", "").strip()
_audio = "demo.wav"
AUDIO_PATH = Path(_audio) if _audio else Path()

if AUDIO_PATH.is_file():
    y, sr = librosa.load(AUDIO_PATH, mono=True, sr=None)
    print(f"Loaded: {AUDIO_PATH.resolve()}, sr={sr} Hz, samples={len(y)}, duration={len(y) / sr:.2f} s")
else:
    sr = 16000
    t = np.linspace(0.0, 2.0, int(2 * sr), endpoint=False, dtype=np.float32)
    y = (0.2 * np.sin(2.0 * np.pi * 200.0 * t)).astype(np.float32)
    print("No valid AUDIO_PATH: using 2 s synthetic 200 Hz tone. Export AUDIO_PATH=/path/to/file.wav for real audio.")

Loaded: /home/luusamp/crossroads/voice-anonymization/notebooks/demo.wav, sr=16000 Hz, samples=83201, duration=5.20 s


In [2]:
# Low-pass blurring only (§2.2.1): 500 Hz downsample then resample to 16 kHz
y_lp, sr_lp = low_pass_blur(y, sr)
print(f"After low_pass_blur: sr={sr_lp} Hz, samples={len(y_lp)}")
display(Audio(y_lp, rate=sr_lp))

After low_pass_blur: sr=16000 Hz, samples=83232


In [3]:
# MFCC inversion only (§2.2.2): first 5 MFCCs → mel → Griffin–Lim
y_mfcc = mfcc_inversion_blur(y, sr)
print(f"After mfcc_inversion_blur: samples={len(y_mfcc)} (may differ from input length)")
display(Audio(y_mfcc, rate=sr))

After mfcc_inversion_blur: samples=82944 (may differ from input length)


In [4]:
# Pipeline: low-pass then MFCC inversion (both stages)
y_lp2, sr_lp2 = low_pass_blur(y, sr)
y_chain = mfcc_inversion_blur(y_lp2, float(sr_lp2))
print(f"Chain: sr_out={sr_lp2} Hz, final samples={len(y_chain)}")
display(Audio(y_chain, rate=sr_lp2))

Chain: sr_out=16000 Hz, final samples=82944
